# Storm Damage — Model Comparison
Upload: `storm_featured.parquet` + `classifier.pkl`

Compares: XGBoost tuned vs XGBoost baseline vs Random Forest vs Logistic Regression
All trained on same X_res, evaluated on same real X_test.

In [ ]:
!pip install xgboost imbalanced-learn scikit-learn joblib -q

In [ ]:
import os, json, time, joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

os.makedirs('models', exist_ok=True)
print('Imports OK')

## CELL 3 — Load + Encode + Split + SMOTE
Same pipeline as training notebook.

In [ ]:
df = pd.read_parquet('storm_featured.parquet')
print(f'Shape: {df.shape}')
print(f'STATE dtype: {df["STATE"].dtype}  ← must be object')

# Encode
for col in ['STATE', 'EVENT_TYPE', 'MONTH_NAME']:
    df[col] = LabelEncoder().fit_transform(df[col])

FEATURE_COLS = [
    'STATE', 'MONTH_NAME', 'EVENT_TYPE',
    'INJURIES_DIRECT', 'INJURIES_INDIRECT',
    'DEATHS_DIRECT', 'DEATHS_INDIRECT',
    'MAGNITUDE', 'duration_min',
    'season', 'state_avg_damage', 'event_avg_damage',
]

X = df[FEATURE_COLS].values.astype(np.float32)
y = df['damage_tier'].values

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE on train only
print('Running SMOTE...')
X_res, y_res = SMOTE(random_state=42).fit_resample(X_train, y_train)
print(f'X_res: {X_res.shape}  X_test: {X_test.shape}')
print(f'Test distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}')

## CELL 4 — Load Tuned XGBoost
Upload `models/classifier.pkl` — this is the already-trained tuned model. No retraining needed.

In [ ]:
xgb_tuned = joblib.load('models/classifier.pkl')
print(f'Loaded tuned XGBoost: {xgb_tuned.get_params()}')

## CELL 5 — Train Baseline XGBoost

In [ ]:
print('Training XGBoost baseline...')
start = time.time()
xgb_base = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                          random_state=42, eval_metric='mlogloss', device='cuda')
xgb_base.fit(X_res, y_res)
print(f'Done in {(time.time()-start)/60:.1f} min')

## CELL 6 — Train Random Forest

In [ ]:
print('Training Random Forest...')
start = time.time()
rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                             random_state=42, n_jobs=-1)
rf.fit(X_res, y_res)
print(f'Done in {(time.time()-start)/60:.1f} min')
joblib.dump(rf, 'models/rf.pkl')
print('Saved: models/rf.pkl')

## CELL 7 — Train Logistic Regression

In [ ]:
print('Training Logistic Regression...')
start = time.time()
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_res, y_res)
print(f'Done in {(time.time()-start)/60:.1f} min')

## CELL 8 — Compare All Models on Real Test Set

In [ ]:
TIER_NAMES = ['None', 'Low', 'Medium', 'High']

models = {
    'XGBoost_tuned':    xgb_tuned,
    'XGBoost_untuned':  xgb_base,
    'RandomForest':     rf,
    'LogisticRegression': lr,
}

results = {}
for name, model in models.items():
    preds = model.predict(X_test)
    f1 = f1_score(y_test, preds, average='weighted')
    results[name] = round(f1, 4)
    print(f'\n── {name} (F1={f1:.4f}) ──')
    print(classification_report(y_test, preds, target_names=TIER_NAMES))

print('\n── Summary ──')
for name, f1 in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f'{name:25s}  weighted F1: {f1:.4f}')

## CELL 9 — Save Updated metrics.json

In [ ]:
winner = max(results, key=results.get)

metrics = {
    **results,
    'winner': winner,
    'best_params': xgb_tuned.get_params(),
}

with open('models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Winner: {winner}')
print(f'Saved: models/metrics.json')
print(json.dumps(metrics, indent=2))

## CELL 10 — Download

In [ ]:
from google.colab import files
files.download('models/metrics.json')
files.download('models/rf.pkl')
print('Replace api/models/metrics.json with new download')